In [53]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# Cargar el dataset preprocesado
df = pd.read_csv("C:/Users/basti/proyecto_cd/data/processed/spotify_dataset_processed.csv", sep=",")

# Definir las características acústicas que usará el modelo
features = ['danceability', 'energy', 'loudness', 'speechiness', 
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# Crear un DataFrame solo con estas características (nuestro "espacio de 9 dimensiones")
X = df[features]

In [54]:
# Inicializar el modelo KNN No Supervisado
# n_neighbors: Queremos 6 vecinos (la canción original + las 5 recomendaciones)
# metric: Usaremos la distancia del coseno
# algorithm: 'brute' o 'auto'. 'brute' calcula todas las distancias, es seguro para este tamaño.
knn_model = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='brute')

# Ajustar (entrenar) el modelo con nuestro espacio de características
print("Entrenando el modelo espacial KNN...")
knn_model.fit(X)
print("¡Modelo entrenado exitosamente!")

Entrenando el modelo espacial KNN...
¡Modelo entrenado exitosamente!


In [55]:
def buscar_cancion_id(nombre, dataset, artista=None):
    """
    Busca el track_id por nombre exacto y por artista de forma parcial.
    Ej: Si el artista es 'Bizarrap', coincidirá con 'Shakira, Bizarrap'.
    """
    # 1. Filtramos primero por el nombre de la canción (exacto, sin importar mayúsculas)
    filtro_nombre = dataset['track_name'].str.lower() == nombre.lower()
    resultado = dataset[filtro_nombre]
    
    if resultado.empty:
        print(f"❌ No se encontró ninguna canción llamada '{nombre}' en el dataset.")
        return None
    
    # 2. Si el usuario ingresó un artista, aplicamos el filtro parcial
    if artista:
        # Usamos contains para coincidencia parcial
        filtro_artista = resultado['artists'].str.contains(artista, case=False, na=False)
        resultado = resultado[filtro_artista]
        
        if resultado.empty:
            print(f"❌ Se encontró la canción '{nombre}', pero NO asociada al artista '{artista}'.")
            return None
            
    # 3. Si hay más de un resultado (ej. remasterizaciones o duplicados no filtrados)
    if len(resultado) > 1:
        artista_encontrado = resultado.iloc[0]['artists']
        print(f"⚠️ Aviso: Hay múltiples versiones. Usando la primera: {nombre} - {artista_encontrado}")
        
    return resultado.iloc[0]['track_id']

In [56]:
def iniciar_app_recomendacion(df, knn_model, features_list):
    """
    Inicia una interfaz interactiva por consola para probar los modelos en vivo.
    """
    print("=" * 65)
    print("🎧 BIENVENIDO AL SISTEMA DE RECOMENDACIÓN MUSICAL (CRISP-DM) 🎧")
    print("=" * 65)
    
    while True:
        print("\n" + "-" * 65)
        print("(Escribe 'salir' en el nombre de la canción para terminar la prueba)")
        
        # Recibir entradas del usuario
        entrada_cancion = input("🎵 Ingresa el nombre de la canción: ").strip()
        
        if entrada_cancion.lower() == 'salir':
            print("\n¡Gracias por probar el sistema! Cerrando... 👋")
            break
            
        entrada_artista = input("🎤 Ingresa el artista (o presiona Enter para omitir): ").strip()
        
        # Si el usuario solo presiona Enter, lo dejamos como None
        if entrada_artista == "":
            entrada_artista = None
            
        print("\nProcesando solicitud...")
        
        # 1. Ejecutar la búsqueda universal
        track_id_encontrado = buscar_cancion_id(entrada_cancion, df, entrada_artista)
        
        if track_id_encontrado:
            print("=" * 65)
            print("🚀 RESULTADOS DE LA BÚSQUEDA Y EVALUACIÓN DE KPIs 🚀")
            print("=" * 65)
            
           # --- EJECUTAR KNN AUTO-EVALUADO ---
            print("\n" + "*" * 20 + " MODELO KNN " + "*" * 20)
            recomendar_por_nombre_knn_y_evaluar(
                nombre_cancion=entrada_cancion, 
                modelo=knn_model, 
                dataframe=df, 
                features_list=features_list, 
                artista=entrada_artista
            )
            # -------------------------------------------------------------
            # NOTA PARA EL NOTEBOOK DE COMPARACIÓN:
            # Si estás en el notebook donde tienen los 3 modelos integrados, 
            # aquí puedes descomentar las líneas de abajo para correr todos a la vez.
            # -------------------------------------------------------------
            
            # print("\n" + "*" * 20 + " MODELO K-MEANS " + "*" * 20)
            # probar_kmeans_por_nombre(entrada_cancion, df, features_list, artista=entrada_artista)
            
            # print("\n" + "*" * 20 + " MODELO DBSCAN " + "*" * 20)
            # probar_dbscan_por_nombre(entrada_cancion, df, features_list, artista=entrada_artista)

In [60]:
import numpy as np
import pandas as pd
import time
import sys

def generar_explicacion_acustica(vector_semilla, vector_recomendado, features_list):
    """
    Compara dos vectores acústicos y identifica las 3 variables con menor diferencia
    para justificar la recomendación al usuario.
    """
    # Calculamos la diferencia absoluta por cada característica
    diferencias = np.abs(vector_semilla.values[0] - vector_recomendado.values[0])
    
    # Mapeo de nombres técnicos a nombres amigables para el usuario
    traduccion = {
        'danceability': 'Bailabilidad',
        'energy': 'Nivel de Energía',
        'loudness': 'Volumen (Loudness)',
        'speechiness': 'Presencia Vocal',
        'acousticness': 'Acústica',
        'instrumentalness': 'Instrumentalidad',
        'liveness': 'Sensación de Vivo',
        'valence': 'Positividad Musical',
        'tempo': 'Ritmo (Tempo)'
    }
    
    # Crear diccionario de diferencias y ordenar
    dif_dict = {features_list[i]: diferencias[i] for i in range(len(features_list))}
    top_3 = sorted(dif_dict.items(), key=lambda item: item[1])[:3]
    
    # Formatear la explicación
    vars_explicacion = [traduccion[var[0]] for var in top_3]
    return f"💡 Recomendada por su similitud en: {vars_explicacion[0]}, {vars_explicacion[1]} y {vars_explicacion[2]}."

# Asegurarnos de tener el evaluador importado
try:
    from evaluador import evaluar_modelo_recomendacion
except ImportError:
    print("⚠️ Recuerda importar evaluar_modelo_recomendacion desde src/evaluador.py")

def recomendar_canciones_knn(track_id_semilla, modelo, dataframe, features_list, top_k=5):
    """
    Motor de recomendación puro. Busca los vecinos y genera explicaciones.
    NO evalúa métricas, solo retorna los datos para que sean evaluados superiormente.
    """
    try:
        idx_semilla = dataframe.index[dataframe['track_id'] == track_id_semilla].tolist()[0]
    except IndexError:
        print(f"❌ Error: El ID '{track_id_semilla}' no existe en el dataset.")
        return None, None

    vector_semilla = dataframe.iloc[[idx_semilla]][features_list]
    nombre_semilla = dataframe.iloc[idx_semilla]['track_name']
    artista_semilla = dataframe.iloc[idx_semilla]['artists']
    
    print(f"🔍 Generando recomendaciones para: '{nombre_semilla}' de {artista_semilla}")
    print("-" * 70)

    distancias, indices = modelo.kneighbors(vector_semilla, n_neighbors=top_k + 1)

    lista_recomendaciones = []
    indices_recomendados = []
    
    for i in range(1, len(distancias[0])):
        idx_vecino = indices[0][i]
        distancia_matematica = distancias[0][i]
        
        track_rec = dataframe.iloc[idx_vecino]
        vector_rec = dataframe.iloc[[idx_vecino]][features_list]
        
        explicacion = generar_explicacion_acustica(vector_semilla, vector_rec, features_list)
        
        print(f"{i}. 🎵 {track_rec['track_name']} - {track_rec['artists']}")
        print(f"   {explicacion}")
        print(f"   (Distancia del Coseno: {distancia_matematica:.4f})\n")
        
        lista_recomendaciones.append({
            'track_name': track_rec['track_name'],
            'artists': track_rec['artists'],
            'distancia': distancia_matematica,
            'explicacion': explicacion
        })
        
        indices_recomendados.append(idx_vecino)
        
    print("-" * 70)
    
    # Retornar los vectores acústicos puros
    vectores_df = dataframe.iloc[indices_recomendados][features_list]
    
    return lista_recomendaciones, vectores_df

In [58]:
def recomendar_por_nombre_knn(nombre_cancion, modelo, dataframe, features_list, artista=None, top_k=5):
    # 1. Obtener el track_id mediante búsqueda por nombre
    track_id = buscar_cancion_id(nombre_cancion, dataframe, artista)
    
    if track_id:
        # 2. Llamar a la función que ya construimos anteriormente con el ID encontrado
        return recomendar_canciones_knn(track_id, modelo, dataframe, features_list, top_k)
    return None, None

# Prueba:
# recomendar_por_nombre_knn("Love Like This", knn_model, df, features, artista="Ben Rector")

In [ ]:
# Iniciar la interfaz de prueba en vivo
iniciar_app_recomendacion(df, knn_model, features)

🎧 BIENVENIDO AL SISTEMA DE RECOMENDACIÓN MUSICAL (CRISP-DM) 🎧

-----------------------------------------------------------------
(Escribe 'salir' en el nombre de la canción para terminar la prueba)


🎵 Ingresa el nombre de la canción:  Let Me Love You
🎤 Ingresa el artista (o presiona Enter para omitir):  



Procesando solicitud...
🚀 RESULTADOS DE LA BÚSQUEDA Y EVALUACIÓN DE KPIs 🚀

******************** MODELO KNN ********************
🔍 Generando recomendaciones para: 'Let Me Love You' de DJ Snake;Justin Bieber
----------------------------------------------------------------------
1. 🎵 Kinderen voor Kinderen Tune - Kinderen voor Kinderen
   💡 Recomendada por su similitud en: Presencia Vocal, Instrumentalidad y Ritmo (Tempo).
   (Distancia del Coseno: 0.0009)

2. 🎵 Bloom - Lucas Estrada;TWOPILOTS
   💡 Recomendada por su similitud en: Instrumentalidad, Bailabilidad y Sensación de Vivo.
   (Distancia del Coseno: 0.0016)

3. 🎵 Pródigo - Casa Worship;Léo Brandão;MOVER
   💡 Recomendada por su similitud en: Instrumentalidad, Positividad Musical y Presencia Vocal.
   (Distancia del Coseno: 0.0016)

4. 🎵 Lost & Found - Embody;Claire Ridgely
   💡 Recomendada por su similitud en: Nivel de Energía, Instrumentalidad y Presencia Vocal.
   (Distancia del Coseno: 0.0018)

5. 🎵 Yaşamak Öldürür, Pt. 1 - Jo

🎵 Ingresa el nombre de la canción:  selfless
🎤 Ingresa el artista (o presiona Enter para omitir):  



Procesando solicitud...
🚀 RESULTADOS DE LA BÚSQUEDA Y EVALUACIÓN DE KPIs 🚀

******************** MODELO KNN ********************
🔍 Generando recomendaciones para: 'Selfless' de The Strokes
----------------------------------------------------------------------
1. 🎵 Therapist^2 - KÅRP
   💡 Recomendada por su similitud en: Positividad Musical, Instrumentalidad y Presencia Vocal.
   (Distancia del Coseno: 0.0014)

2. 🎵 Trampoline - Jauz Remix - SHAED;Jauz
   💡 Recomendada por su similitud en: Instrumentalidad, Presencia Vocal y Sensación de Vivo.
   (Distancia del Coseno: 0.0017)

3. 🎵 I Will Return - Skylar Grey
   💡 Recomendada por su similitud en: Presencia Vocal, Instrumentalidad y Bailabilidad.
   (Distancia del Coseno: 0.0023)

4. 🎵 Never Enough - Divide Music
   💡 Recomendada por su similitud en: Presencia Vocal, Positividad Musical y Instrumentalidad.
   (Distancia del Coseno: 0.0025)

5. 🎵 1,000 Names - Phil Wickham
   💡 Recomendada por su similitud en: Instrumentalidad, Presenci

...


Procesando solicitud...
🚀 RESULTADOS DE LA BÚSQUEDA Y EVALUACIÓN DE KPIs 🚀

******************** MODELO KNN ********************
🔍 Generando recomendaciones para: 'Quién Fuera' de Los Bunkers
----------------------------------------------------------------------
1. 🎵 Bend & Break - Keane
   💡 Recomendada por su similitud en: Acústica, Instrumentalidad y Ritmo (Tempo).
   (Distancia del Coseno: 0.0006)

2. 🎵 Ay, Que Dios Boludo - Divididos
   💡 Recomendada por su similitud en: Acústica, Ritmo (Tempo) y Instrumentalidad.
   (Distancia del Coseno: 0.0008)

3. 🎵 Astronaut - Simple Plan;Julian Emery
   💡 Recomendada por su similitud en: Volumen (Loudness), Instrumentalidad y Acústica.
   (Distancia del Coseno: 0.0009)

4. 🎵 Christmas All Over Again - The Goo Goo Dolls
   💡 Recomendada por su similitud en: Positividad Musical, Acústica y Presencia Vocal.
   (Distancia del Coseno: 0.0013)

5. 🎵 Otherside - Red Hot Chili Peppers
   💡 Recomendada por su similitud en: Instrumentalidad, Acústica

🎵 Ingresa el nombre de la canción:  Where Are Ü Now 
🎤 Ingresa el artista (o presiona Enter para omitir):  



Procesando solicitud...
❌ No se encontró ninguna canción llamada 'Where Are Ü Now' en el dataset.

-----------------------------------------------------------------
(Escribe 'salir' en el nombre de la canción para terminar la prueba)


🎵 Ingresa el nombre de la canción:  Where Are Ü Now (with Justin Bieber)
🎤 Ingresa el artista (o presiona Enter para omitir):  



Procesando solicitud...
🚀 RESULTADOS DE LA BÚSQUEDA Y EVALUACIÓN DE KPIs 🚀

******************** MODELO KNN ********************
🔍 Generando recomendaciones para: 'Where Are Ü Now (with Justin Bieber)' de Jack Ü;Skrillex;Diplo;Justin Bieber
----------------------------------------------------------------------
1. 🎵 Heavenly Side - ILLENIUM;Matt Maeson
   💡 Recomendada por su similitud en: Instrumentalidad, Presencia Vocal y Positividad Musical.
   (Distancia del Coseno: 0.0005)

2. 🎵 Khwaab - Thaikkudam Bridge
   💡 Recomendada por su similitud en: Instrumentalidad, Ritmo (Tempo) y Positividad Musical.
   (Distancia del Coseno: 0.0005)

3. 🎵 Worthy of My Song - Phil Wickham
   💡 Recomendada por su similitud en: Instrumentalidad, Ritmo (Tempo) y Positividad Musical.
   (Distancia del Coseno: 0.0007)

4. 🎵 Vi Vokser - The Minds Of 99
   💡 Recomendada por su similitud en: Positividad Musical, Sensación de Vivo y Nivel de Energía.
   (Distancia del Coseno: 0.0008)

5. 🎵 What's This Life Fo